# Correlation and Relationship Analysis

Analysing relationships between operational variables and rider-experience outcomes.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.analysis.relationships import (
    compute_correlations,
    compare_relationships_by_demand,
    compare_relationships_by_city,
)

In [ ]:
# Load and engineer features
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows')

## Overall Correlations

In [ ]:
correlations = compute_correlations(df)
corr_df = pd.DataFrame([vars(c) for c in correlations])
corr_df.sort_values('coefficient', key=abs, ascending=False).round(3)

## High-Demand vs Normal-Demand

In [ ]:
high_corr, normal_corr = compare_relationships_by_demand(df)
print('High-demand correlations:')
for c in high_corr:
    if abs(c.coefficient) > 0.2:
        print(f'  {c.var1} <-> {c.var2}: {c.coefficient:.3f}')

print('\nNormal-demand correlations:')
for c in normal_corr:
    if abs(c.coefficient) > 0.2:
        print(f'  {c.var1} <-> {c.var2}: {c.coefficient:.3f}')

## City-Level Relationships

In [ ]:
# Demand-supply ratio vs surge by city
city_corr = compare_relationships_by_city(df, 'demand_supply_ratio', 'surge_multiplier')
for c in city_corr:
    print(f'{c.city}: r={c.coefficient:.3f} (p={c.p_value:.4f})')